## **Machine Learning Aplicado a las Finanzas** 🚀
### **HW Sesión 8: Random Forest — Ensembles, Decorrelación y Aplicaciones Financieras**

Andrés C. Medina Sanhueza

Senior Data Scientist Engineer

anmedinas@gmail.com

---

### Consideraciones Previas

* Tarea es **Individual**
* Fecha de Entrega: **Jueves 16 Jul 23:59** (cualquier commit posterior descontará puntos)
* El notebook debe correr de arriba abajo sin errores (`Kernel → Restart & Run All`)
* No se admiten `pass` pendientes ni celdas vacías donde se espera código
* **Regla de oro:** si se detecta data leakage en alguna solución (uso de información futura en la construcción de features o en la partición train/val/test), esa parte se evalúa con nota 1.0
* Deben crear una rama nueva en su repositorio `feat/hw06` y trabajar la tarea en esa rama. El notebook debe quedar en la carpeta `hw/`
* **Archivo a entregar:** URL del repositorio GitHub enviada a `anmedinas@gmail.com`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, accuracy_score, r2_score, mean_squared_error,
    precision_recall_curve, roc_curve
)
from sklearn.inspection import permutation_importance
import yfinance as yf
import warnings

sns.set_style("dark")
warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.float_format", "{:.4f}".format)

---
## Parte 1 — Bagging y Random Forest desde Cero *(25 pts)*

### Contexto

En la sesión 8 vimos que Random Forest se construye en dos capas:

1. **Bagging (Breiman, 1996):** se generan $B$ datasets bootstrap $\mathcal{D}^{*(b)}$ (muestreo con reemplazo, mismo tamaño que $\mathcal{D}$) y se ajusta un árbol por cada uno. La predicción final agrega los $B$ árboles (promedio en regresión, voto/promedio de probabilidad en clasificación). Esto reduce varianza sin aumentar sesgo.
2. **Feature subsampling (Breiman, 2001):** en cada split de cada árbol, solo se evalúa un subconjunto aleatorio de $m < p$ features. Esto decorrelaciona los árboles — sin este paso, un feature dominante haría que casi todos los árboles bootstrap elijan el mismo primer split, y el promedio de árboles altamente correlacionados reduce varianza mucho menos que el promedio de árboles independientes:

$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^B \hat{f}_b(x)\right) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

donde $\rho$ es la correlación promedio entre pares de árboles. Cuando $B \to \infty$, el primer término ($\rho\sigma^2$) **no desaparece** — por eso decorrelacionar (reducir $\rho$) importa tanto como promediar.

En esta parte implementarás bagging + Random Forest **desde cero a nivel de ensemble**, reutilizando `DecisionTreeClassifier`/`DecisionTreeRegressor` de sklearn como bloque base (no reimplementas CART, sino el mecanismo de bootstrap + agregación + OOB que es el objeto de esta sesión). El parámetro `max_features` de `DecisionTreeClassifier` ya implementa el feature subsampling por split, así que fijarlo en cada árbol bootstrap replica exactamente el mecanismo de Random Forest.

### Datos de trabajo

La celda siguiente descarga precios de un activo real y construye features técnicas *anti-leakage* (todas con `.shift(1)` o rolling sobre datos ya rezagados) para predecir la **dirección del retorno del día siguiente** (clasificación binaria). Puedes cambiar `TICKER` por otro activo si lo prefieres, pero mantén el período para que los resultados de crisis (Parte 4) sean comparables.

In [ ]:
TICKER = "AAPL"

raw = yf.download(TICKER, start="2015-01-01", end="2024-01-01", progress=False, auto_adjust=True)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

close = raw["Close"]
ret = close.pct_change()

feat = pd.DataFrame(index=raw.index)
feat["ret_lag1"] = ret.shift(1)
feat["ret_lag2"] = ret.shift(2)
feat["ret_lag5"] = ret.shift(1).rolling(5).sum()
feat["vol_20"]   = ret.shift(1).rolling(20).std() * np.sqrt(252)
feat["mom_10"]   = close.shift(1) / close.shift(11) - 1
delta = close.diff()
gain = delta.clip(lower=0).shift(1).rolling(14).mean()
loss = (-delta.clip(upper=0)).shift(1).rolling(14).mean()
feat["rsi_14"]   = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
feat["vol_chg"]  = raw["Volume"].pct_change().shift(1)
feat["target"]   = (ret.shift(-1) > 0).astype(int)   # 1 = sube el dia siguiente
feat = feat.dropna()

FEATURES = ["ret_lag1", "ret_lag2", "ret_lag5", "vol_20", "mom_10", "rsi_14", "vol_chg"]

X = feat[FEATURES].values
y = feat["target"].values

split = int(len(feat) * 0.7)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"{TICKER}: {feat.shape[0]} obs | {len(FEATURES)} features")
print(f"Train: {X_train.shape[0]} obs ({feat.index[0].date()} – {feat.index[split-1].date()})")
print(f"Test:  {X_test.shape[0]} obs ({feat.index[split].date()} – {feat.index[-1].date()})")
print(f"Tasa de subida — train: {y_train.mean():.3%} | test: {y_test.mean():.3%}")

### 1.1 — Bagging Forest: Bootstrap + Agregación *(13 pts)*

Implementa dos funciones:

- `bagging_forest_fit(X, y, n_estimators, max_features, task, max_depth, min_samples_leaf, random_state)`: para cada uno de los `n_estimators` árboles, genera un bootstrap sample (con reemplazo, mismo tamaño que `X`), ajusta un `DecisionTreeClassifier` o `DecisionTreeRegressor` (según `task`) con ese `max_features`, y registra los **índices out-of-bag** (las filas que no aparecieron en ese bootstrap). Debe retornar una lista de tuplas `(árbol_ajustado, índices_oob)`.
- `bagging_forest_predict(trees_info, X, task)`: agrega las predicciones de todos los árboles — promedio de `predict_proba` en clasificación (y umbral 0.5 para la clase), promedio de `predict` en regresión.

**Requisitos:**
1. Implementar ambas funciones (usa `np.random.RandomState(random_state)` para las semillas de bootstrap, con una semilla distinta — pero reproducible — por árbol)
2. Entrenar con `n_estimators=200`, `max_features="sqrt"`, `min_samples_leaf=10` sobre `X_train, y_train`
3. Verificar que la **fracción promedio de observaciones OOB por árbol** esté cerca de $1 - 1/e \approx 0.368$ (justifica por qué, matemáticamente, ese es el límite esperado)
4. Comparar accuracy y AUC-ROC en test contra un `RandomForestClassifier` de sklearn con los mismos hiperparámetros — la diferencia debe ser razonablemente pequeña (`< 0.05` en ambas métricas); no esperes un match exacto porque el mecanismo interno de bootstrap de sklearn difiere en detalles de implementación

In [ ]:
def bagging_forest_fit(X, y, n_estimators=100, max_features="sqrt", task="classification",
                        max_depth=None, min_samples_leaf=1, random_state=42):
    '''
    Ajusta un ensemble de bagging (Random Forest) desde cero.

    Parameters
    ----------
    X, y         : arrays de entrenamiento
    n_estimators : int   — numero de arboles bootstrap (B)
    max_features : int, float, str o None — pasado directo al arbol base
    task         : "classification" o "regression"
    max_depth, min_samples_leaf : hiperparametros del arbol base
    random_state : semilla maestra (debe generar semillas reproducibles por arbol)

    Returns
    -------
    trees_info : list de tuplas (arbol_ajustado, oob_idx)
                 oob_idx: array con los indices de X que NO aparecieron en el bootstrap de ese arbol
    '''
    # TODO: implementa los pasos descritos en el enunciado
    pass


def bagging_forest_predict(trees_info, X, task="classification"):
    '''
    Agrega las predicciones de todos los arboles del ensemble.

    Returns
    -------
    Si task == "classification": (pred_clase, proba_clase_1)
    Si task == "regression":      pred (promedio de los arboles)
    '''
    # TODO: implementa la agregacion (voto/promedio para clasificacion, promedio para regresion)
    pass


# === Verificacion ===
# Descomenta cuando hayas implementado ambas funciones

# trees_info = bagging_forest_fit(X_train, y_train, n_estimators=200, max_features="sqrt",
#                                  task="classification", min_samples_leaf=10, random_state=42)
# pred_manual, proba_manual = bagging_forest_predict(trees_info, X_test, task="classification")

# acc_manual = accuracy_score(y_test, pred_manual)
# auc_manual = roc_auc_score(y_test, proba_manual)

# rf_ref = RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=10,
#                                  random_state=42, n_jobs=-1).fit(X_train, y_train)
# acc_ref = rf_ref.score(X_test, y_test)
# auc_ref = roc_auc_score(y_test, rf_ref.predict_proba(X_test)[:, 1])

# oob_fracs = [len(oob) / len(X_train) for _, oob in trees_info]

# print(f"Manual  — acc={acc_manual:.4f}  auc={auc_manual:.4f}")
# print(f"sklearn — acc={acc_ref:.4f}  auc={auc_ref:.4f}")
# print(f"|diff acc|={abs(acc_manual-acc_ref):.4f}  |diff auc|={abs(auc_manual-auc_ref):.4f}")
# print(f"Fraccion OOB promedio: {np.mean(oob_fracs):.4f}  (teorico 1-1/e={1-1/np.e:.4f})")

### 1.2 — Error Out-of-Bag (OOB) Manual *(12 pts)*

Cada árbol del bosque no vio ~36.8% de las observaciones de entrenamiento (sus OOB). Esto permite estimar el error de generalización **sin gastar un set de validación separado**: para cada observación $i$, se agregan solo las predicciones de los árboles que no la vieron durante su bootstrap.

Implementa `oob_score_manual(trees_info, X, y, task)` que:
1. Para cada observación $i$, recolecta las predicciones de todos los árboles cuyo `oob_idx` contiene a $i$
2. Agrega esas predicciones (promedio, igual que en `bagging_forest_predict`)
3. Calcula accuracy (clasificación) o $R^2$ (regresión) solo sobre las observaciones que tuvieron **al menos un árbol OOB** que las cubriera (reporta también qué fracción de observaciones quedó cubierta)

**Requisitos:**
1. Implementar `oob_score_manual` y verificar contra `RandomForestClassifier(oob_score=True).oob_score_` — la diferencia debe ser `< 0.05`
2. Replicar el gráfico de la sesión 8: convergencia del error OOB vs error de test real, barriendo `n_estimators` en `[1, 5, 10, 20, 50, 100, 200, 300]`, sobre los datos reales de `TICKER`
3. Reportar en cuántos árboles (aprox.) el error OOB dejó de mejorar significativamente

In [ ]:
def oob_score_manual(trees_info, X, y, task="classification"):
    '''
    Calcula el error Out-of-Bag agregando, para cada observacion, solo los
    arboles que no la vieron en su bootstrap.

    Returns
    -------
    score    : accuracy (task="classification") o R2 (task="regression")
    coverage : fraccion de observaciones de X que tuvieron >=1 arbol OOB
    '''
    # TODO: implementa los 3 pasos descritos en el enunciado
    pass


# === Verificacion y grafico de convergencia ===
# Descomenta cuando hayas implementado la funcion

# n_est_range = [1, 5, 10, 20, 50, 100, 200, 300]
# oob_errors, test_errors = [], []
# for n_est in n_est_range:
#     trees_n = bagging_forest_fit(X_train, y_train, n_estimators=n_est, max_features="sqrt",
#                                   task="classification", min_samples_leaf=10, random_state=42)
#     oob_acc, coverage = oob_score_manual(trees_n, X_train, y_train, task="classification")
#     pred_n, _ = bagging_forest_predict(trees_n, X_test, task="classification")
#     oob_errors.append(1 - oob_acc)
#     test_errors.append(1 - accuracy_score(y_test, pred_n))

# fig, ax = plt.subplots(figsize=(8, 5))
# ax.plot(n_est_range, oob_errors, "o-", color="darkorange", lw=2, label="OOB error (manual)")
# ax.plot(n_est_range, test_errors, "s-", color="steelblue", lw=2, label="Test error")
# ax.set_xscale("log"); ax.set_xlabel("n_estimators"); ax.set_ylabel("Error (1 - Accuracy)")
# ax.set_title(f"Convergencia OOB vs Test — {TICKER}"); ax.legend()
# plt.tight_layout(); plt.show()

# rf_oob_ref = RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=10,
#                                      oob_score=True, random_state=42, n_jobs=-1).fit(X_train, y_train)
# print(f"sklearn oob_score_ (n=200): {rf_oob_ref.oob_score_:.4f}")

### Preguntas de Interpretación — Parte 1

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿Por qué la fracción esperada de observaciones OOB por árbol converge a $1 - 1/e$ cuando $n \to \infty$? (Pista: probabilidad de que una observación específica NO sea elegida en ninguna de las $n$ extracciones con reemplazo)

> *Completa aquí*

**b)** Compara tu error OOB manual con el error de test real. ¿Son parecidos? ¿Bajo qué supuesto sobre la generación de los datos es válido usar OOB como proxy del error de generalización, y por qué ese supuesto es más frágil en series de tiempo financieras que en datos i.i.d.?

> *Completa aquí*

**c)** En el gráfico de convergencia, ¿a partir de qué `n_estimators` la ganancia adicional es marginal? Relaciona esto con la fórmula $\rho\sigma^2 + \frac{(1-\rho)}{B}\sigma^2$: ¿qué término explica por qué el error nunca llega a cero por más árboles que agregues?

> *Completa aquí*

---
## Parte 2 — Hiperparámetros y Sesgo-Varianza en Datos Reales *(25 pts)*

### Contexto

La sesión 8 mostró curvas de sesgo-varianza (árbol vs Random Forest) y el efecto de `max_features` sobre datos sintéticos. Aquí repites el análisis sobre datos financieros reales, donde además hay que lidiar con la estructura temporal: la validación cruzada estándar (k-fold aleatorio) filtra información del futuro hacia el pasado, así que se usa `TimeSeriesSplit`.

### 2.1 — Curvas de Sesgo-Varianza: Árbol vs Random Forest *(10 pts)*

**Requisitos:**
1. Sobre `X_train, y_train` / `X_test, y_test` del `TICKER` elegido, barre `max_depth` en `range(1, 16)` y ajusta, para cada profundidad: un único `DecisionTreeClassifier` y un `RandomForestClassifier(n_estimators=150, max_features="sqrt")`
2. Grafica accuracy train/test de ambos modelos en dos paneles (como en la sesión 8), marcando con línea vertical el `max_depth` con mejor accuracy de test en cada panel
3. Reporta el gap train-test (varianza) para `max_depth=10` en ambos modelos

In [ ]:
# TODO: barrido de max_depth para arbol individual vs Random Forest sobre datos reales
# depths = range(1, 16)
# tree_train_acc, tree_test_acc = [], []
# rf_train_acc, rf_test_acc = [], []
# for d in depths:
#     ...

# TODO: grafico de 2 paneles (arbol, RF) con train/test accuracy vs max_depth
# TODO: imprime el gap train-test en max_depth=10 para ambos modelos

### 2.2 — Grid Search Walk-Forward y ¿Es Confiable el OOB en Series de Tiempo? *(15 pts)*

**Requisitos:**
1. Implementa un grid search sobre `n_estimators ∈ {50, 100, 200}` y `max_features ∈ {2, 3, "sqrt", 5}`, evaluando cada combinación con `TimeSeriesSplit(n_splits=5, gap=5)` sobre `X_train, y_train` (métrica: AUC-ROC promedio de los folds de validación)
2. Reporta un heatmap del AUC de validación walk-forward para las 12 combinaciones
3. Para cada combinación del grid, entrena también un `RandomForestClassifier(oob_score=True)` sobre **todo** `X_train` y guarda su `oob_score_`
4. Calcula la correlación entre el OOB score y el AUC walk-forward promedio a través de las 12 combinaciones. Interpreta el signo y la magnitud

**Nota conceptual:** el bootstrap de bagging asume que las observaciones son intercambiables (i.i.d.). En series de tiempo financieras, filas cercanas en el tiempo están autocorrelacionadas, así que una observación que "no vio" un árbol (OOB) puede seguir siendo muy parecida a observaciones que sí vio ese árbol — el OOB deja de ser un holdout limpio. Esta parte cuantifica qué tan grave es ese problema en tu dataset.

In [ ]:
# TODO: grid search walk-forward
# tscv = TimeSeriesSplit(n_splits=5, gap=5)
# grid_ne = [50, 100, 200]
# grid_mf = [2, 3, "sqrt", 5]
# results = []
# for ne in grid_ne:
#     for mf in grid_mf:
#         ...  # loop de folds, entrenar, evaluar AUC de validacion
#         ...  # entrenar tambien con oob_score=True sobre todo X_train

# TODO: heatmap AUC walk-forward (filas=n_estimators, columnas=max_features)
# TODO: correlacion entre OOB score y AUC walk-forward promedio (np.corrcoef)
# TODO: selecciona la mejor combinacion segun AUC walk-forward y evaluala en X_test, y_test

### Preguntas de Interpretación — Parte 2

**Responde (mínimo 2 oraciones por pregunta):**

**a)** En la curva de sesgo-varianza, ¿en qué `max_depth` el árbol individual empieza a sobreajustar (cae el accuracy de test)? ¿El Random Forest muestra el mismo patrón de caída? Relaciona la diferencia con el mecanismo de bagging.

> *Completa aquí*

**b)** ¿Qué signo tiene la correlación entre OOB score y AUC walk-forward que obtuviste? Si es baja o negativa, ¿qué implica esto para un data scientist que use el OOB score como criterio único de selección de hiperparámetros en un modelo de trading o credit scoring?

> *Completa aquí*

**c)** ¿Cuál fue la mejor combinación de `max_features`/`n_estimators` según walk-forward? ¿Coincide con la que hubieras elegido usando solo el OOB score? Si no coinciden, ¿cuál usarías en producción y por qué?

> *Completa aquí*

---
## Parte 3 — Feature Importance: MDI vs Permutation Importance *(25 pts)*

### Contexto

**MDI (Mean Decrease Impurity):** para cada feature $j$, promedia la reducción de impureza que produjo $j$ en todos los splits donde fue usado, a través de todos los árboles del bosque:

$$\text{MDI}(j) = \frac{1}{B}\sum_{b=1}^{B}\sum_{t \in \text{nodos de } j} p(t)\,\Delta i(t)$$

MDI es **rápido** (se calcula durante el entrenamiento) pero está **sesgado hacia features de alta cardinalidad** (muchos valores únicos, como precios o variables continuas): más valores únicos implican más puntos de corte posibles, y el árbol puede encontrar splits que reducen impureza *por azar* en la muestra de entrenamiento, incluso si el feature no tiene ninguna relación real con $y$.

**Permutation Importance** corrige este sesgo evaluando el modelo **ya entrenado**: para cada feature $j$, se permutan (baraja) sus valores en un set de evaluación, se mide la caída en el desempeño (AUC, accuracy, etc.), y se repite varias veces para promediar el ruido del barajado. Si $j$ es realmente irrelevante, la caída esperada es ≈ 0 sin importar su cardinalidad, porque el barajado no altera su distribución marginal.

### 3.1 — Permutation Importance desde Cero: Demostración del Sesgo de MDI *(13 pts)*

La celda siguiente genera un dataset sintético de *credit scoring* con 6 features: tres genuinamente informativas (`dti`, `score`, `antiguedad`), una moderadamente informativa pero de **baja cardinalidad** (`n_productos`, entera 0-10), y dos **no informativas** — una de **alta cardinalidad** (`ruido_continuo`, uniforme continua) y una de baja cardinalidad (`ruido_binario`, 0/1). Esto replica, de forma controlada, la demostración de la sesión 8.

Implementa `permutation_importance_manual(model, X, y, metric_fn, n_repeats, random_state)`:
1. Calcula el score base del modelo sobre `(X, y)` con `metric_fn`
2. Para cada feature $j$ y cada repetición: permuta (baraja) la columna $j$ de una copia de `X`, recalcula el score, y guarda `score_base - score_permutado`
3. Retorna la media y desviación estándar de las `n_repeats` caídas, por feature

**Requisitos:**
1. Implementar la función y verificarla contra `sklearn.inspection.permutation_importance` (mismo modelo, mismas repeticiones, misma métrica) — la diferencia máxima entre ambas debe ser `< 0.01`
2. Graficar barras horizontales de MDI (`rf.feature_importances_`) vs tu permutation importance manual, coloreando en rojo `ruido_continuo` y `ruido_binario`
3. Reportar el **ranking** (posición 1 a 6) de `ruido_continuo` según MDI vs según permutation importance — ¿en qué posición debería estar si el modelo estuviera bien calibrado?

In [ ]:
np.random.seed(42)
n_fi = 1500

X_fi = pd.DataFrame({
    "dti":             np.random.normal(0.35, 0.15, n_fi),
    "score":           np.random.normal(650, 80, n_fi),
    "antiguedad":      np.random.exponential(5, n_fi),
    "n_productos":     np.random.poisson(3, n_fi),                 # informativa, baja cardinalidad
    "ruido_continuo":  np.random.uniform(0, 1_000_000, n_fi),       # NO informativa, alta cardinalidad
    "ruido_binario":   (np.random.rand(n_fi) < 0.5).astype(int),   # NO informativa, baja cardinalidad
})
logit_fi = (-2 + 3.0 * X_fi["dti"] - 0.004 * (X_fi["score"] - 650)
            - 0.1 * X_fi["antiguedad"] + 0.05 * X_fi["n_productos"])
p_fi = 1 / (1 + np.exp(-logit_fi))
y_fi = (np.random.rand(n_fi) < p_fi).astype(int)

split_fi = int(n_fi * 0.7)
X_tr_fi, X_te_fi = X_fi.values[:split_fi], X_fi.values[split_fi:]
y_tr_fi, y_te_fi = y_fi[:split_fi], y_fi[split_fi:]
fi_names = X_fi.columns.tolist()

print(f"Tasa de default: {y_fi.mean():.2%}")


def permutation_importance_manual(model, X, y, metric_fn, n_repeats=20, random_state=42):
    '''
    Permutation importance implementada desde cero.

    Parameters
    ----------
    model     : modelo ya entrenado, con predict_proba o predict
    X, y      : datos de evaluacion (tipicamente test)
    metric_fn : funcion metric_fn(y_true, y_score_o_pred) -> float (mayor = mejor)
    n_repeats : numero de barajados por feature

    Returns
    -------
    importances_mean, importances_std : arrays (n_features,)
    '''
    # TODO: implementa los 3 pasos descritos en el enunciado
    pass


# === Verificacion ===
# Descomenta cuando hayas implementado la funcion

# rf_fi = RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=5,
#                                 random_state=42, n_jobs=-1).fit(X_tr_fi, y_tr_fi)
# mdi_fi = pd.Series(rf_fi.feature_importances_, index=fi_names)

# perm_mean, perm_std = permutation_importance_manual(
#     rf_fi, X_te_fi, y_te_fi, roc_auc_score, n_repeats=20, random_state=42)
# perm_manual_fi = pd.Series(perm_mean, index=fi_names)

# perm_sklearn = permutation_importance(rf_fi, X_te_fi, y_te_fi, n_repeats=20,
#                                        random_state=42, scoring="roc_auc", n_jobs=-1)
# perm_ref_fi = pd.Series(perm_sklearn.importances_mean, index=fi_names)

# print(f"Max |manual - sklearn| = {np.abs(perm_manual_fi.values - perm_ref_fi.values).max():.5f}")
# print(f"ruido_continuo — rank MDI: {mdi_fi.rank(ascending=False)['ruido_continuo']:.0f}/6  "
#       f"rank permutation: {perm_manual_fi.rank(ascending=False)['ruido_continuo']:.0f}/6")

# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# colors_fi = ["tomato" if "ruido" in f else "steelblue" for f in mdi_fi.sort_values().index]
# mdi_fi.sort_values().plot(kind="barh", ax=axes[0], color=colors_fi)
# axes[0].set_title("MDI")
# colors_perm = ["tomato" if "ruido" in f else "steelblue" for f in perm_manual_fi.sort_values().index]
# perm_manual_fi.sort_values().plot(kind="barh", ax=axes[1], color=colors_perm)
# axes[1].set_title("Permutation Importance (manual)")
# plt.tight_layout(); plt.show()

### 3.2 — Estabilidad Temporal de Feature Importance con Intervalos de Confianza *(12 pts)*

Ya validaste tu función de permutation importance. Ahora aplícala a los datos reales de `TICKER` (Parte 1) para responder una pregunta distinta: **¿las features importantes hoy lo siguen siendo en el futuro?**

**Requisitos:**
1. Define al menos 4 ventanas de entrenamiento expansivas (por ejemplo, usando el 40%, 55%, 70% y 85% de `feat` como corte de entrenamiento)
2. Para cada ventana, entrena un `RandomForestClassifier` y calcula tu `permutation_importance_manual` sobre un bloque de validación inmediatamente posterior al corte (sin usar datos futuros a esa ventana)
3. Para capturar la incertidumbre de la estimación (no solo el punto), repite el cálculo de importancia con **5 semillas distintas** (`random_state`) por ventana y reporta el intervalo `[percentil 10, percentil 90]` de la importancia de cada feature en cada ventana
4. Grafica un heatmap de importancia promedio (ventanas × features) y calcula el coeficiente de variación $CV_j = \text{std}_j/\text{mean}_j$ a través de las ventanas para cada feature; marca en rojo las features con $CV_j > 0.5$

In [ ]:
# TODO: importancia temporal con intervalos de confianza via multiples semillas
# window_fracs = [0.40, 0.55, 0.70, 0.85]
# seeds = [0, 1, 2, 3, 4]
# fi_windows = {}
# for frac in window_fracs:
#     cut = int(len(feat) * frac)
#     val_end = min(cut + 250, len(feat))  # bloque de validacion posterior al corte
#     ...  # entrenar RF con cada seed sobre feat[:cut], evaluar importancia en feat[cut:val_end]
#     ...  # guardar percentil 10/50/90 por feature

# TODO: heatmap ventanas x features (importancia promedio)
# TODO: coeficiente de variacion (CV_j) por feature a traves de ventanas, marcar CV_j > 0.5

### Preguntas de Interpretación — Parte 3

**Responde (mínimo 2 oraciones por pregunta):**

**a)** En 3.1, ¿qué feature quedó mejor rankeada por MDI que por permutation importance de forma más llamativa? Explica el mecanismo por el cual el árbol pudo "engañarse" con esa feature durante el entrenamiento.

> *Completa aquí*

**b)** Compara el ranking de `n_productos` (informativa, baja cardinalidad) contra `ruido_continuo` (no informativa, alta cardinalidad) en ambos métodos. ¿Qué error de decisión cometerías si usaras solo MDI para seleccionar features en un modelo de producción?

> *Completa aquí*

**c)** En 3.2, ¿qué features tuvieron mayor coeficiente de variación entre ventanas? ¿Confiarías en esas features como "backbone" de un modelo de trading que se re-entrena mensualmente? Justifica.

> *Completa aquí*

---
## Parte 4 — Aplicación Financiera: Eventos de Cola y Límite de Extrapolación *(25 pts)*

### Contexto

Random Forest hace predicciones **constantes por región**: cada hoja del árbol predice el promedio (o la clase mayoritaria) de las observaciones de entrenamiento que cayeron en esa región. Esto tiene una consecuencia crítica en finanzas: **el modelo nunca puede predecir un valor fuera del rango de $y$ visto en entrenamiento**, sin importar cuán extremos sean los features de entrada. Un modelo lineal (Ridge), en cambio, sí puede extrapolar — para bien o para mal.

Esta parte tiene dos aplicaciones sobre el período **2015–2024**, que incluye la crisis COVID-19 de marzo 2020 — un evento de cola real que ningún modelo entrenado solo con datos previos a 2020 pudo haber visto.

### 4.1 — Clasificación de Eventos de Cola con Walk-Forward *(13 pts)*

**Requisitos:**
1. Define el target `target_tail = 1` si el retorno del día siguiente cae en el percentil 5 más bajo de la muestra completa (evento de cola / crash-like), usando las mismas `FEATURES` de la Parte 1
2. Implementa un walk-forward de al menos 6 ventanas (expansión del set de entrenamiento, validación en bloques sucesivos) que necesariamente incluya una ventana de validación que contenga marzo de 2020
3. En cada ventana, entrena un `RandomForestClassifier(class_weight="balanced")` y una `LogisticRegression(class_weight="balanced")` de referencia; reporta AUC-ROC de ambos por ventana
4. Grafica la evolución del AUC de ambos modelos a través de las ventanas, marcando la ventana que contiene la crisis COVID

In [ ]:
feat["target_tail"] = (ret.shift(-1) <= ret.quantile(0.05)).astype(int)
feat_tail = feat.dropna()
X_tail = feat_tail[FEATURES].values
y_tail = feat_tail["target_tail"].values
print(f"Tasa de eventos de cola: {y_tail.mean():.2%}")

# TODO: walk-forward de al menos 6 ventanas, comparando RF vs Logistic Regression
# n_windows = 6
# fold_size = len(feat_tail) // (n_windows + 2)
# for w in range(n_windows):
#     ...  # definir train_end, val_end; entrenar ambos modelos; guardar AUC y fechas

# TODO: grafico de AUC_RF y AUC_LR por ventana, marcando la ventana que cubre marzo 2020

### 4.2 — RF No Extrapola: Evidencia con Volatilidad Realizada en la Crisis COVID-19 *(12 pts)*

**Requisitos:**
1. Construye `rv_20` (volatilidad realizada anualizada, ventana de 20 días) y features rezagadas `rv_lag1`, `rv_lag5`, `rv_lag22`
2. Entrena con datos **estrictamente anteriores a 2020-01-01** un `RandomForestRegressor` y un `Ridge`, y predice sobre la ventana **2020-01-01 a 2020-07-01** (incluye el shock de marzo)
3. Reporta: `max(y_train)` vs `max(y_test)` vs `max(predicción RF)` vs `max(predicción Ridge)` — confirma que el máximo de la predicción RF nunca supera el máximo visto en entrenamiento
4. Grafica la serie de volatilidad real vs predicha (ambos modelos) durante la ventana de test, y calcula el RMSE de cada modelo restringido a los días donde `y_test` superó el máximo histórico de entrenamiento (los días de extrapolación pura)

In [ ]:
rv20 = ret.rolling(20).std() * np.sqrt(252)
feat_rv = pd.DataFrame(index=raw.index)
feat_rv["rv_lag1"]  = rv20.shift(1)
feat_rv["rv_lag5"]  = rv20.shift(5)
feat_rv["rv_lag22"] = rv20.shift(22)
feat_rv["target_rv"] = rv20
feat_rv = feat_rv.dropna()

RV_FEATURES = ["rv_lag1", "rv_lag5", "rv_lag22"]
train_mask_rv = feat_rv.index < "2020-01-01"
test_mask_rv  = (feat_rv.index >= "2020-01-01") & (feat_rv.index < "2020-07-01")

X_train_rv = feat_rv.loc[train_mask_rv, RV_FEATURES].values
y_train_rv = feat_rv.loc[train_mask_rv, "target_rv"].values
X_test_rv  = feat_rv.loc[test_mask_rv,  RV_FEATURES].values
y_test_rv  = feat_rv.loc[test_mask_rv,  "target_rv"].values
dates_test_rv = feat_rv.loc[test_mask_rv].index

print(f"Train: {len(y_train_rv)} obs (< 2020-01-01) | Test: {len(y_test_rv)} obs (2020-01-01 a 2020-07-01)")

# TODO: entrenar RandomForestRegressor y Ridge sobre (X_train_rv, y_train_rv)
# TODO: predecir sobre X_test_rv con ambos modelos
# TODO: reportar max(y_train_rv), max(y_test_rv), max(pred_rf), max(pred_ridge)
# TODO: grafico serie temporal: y_test_rv (verdad) vs pred_rf vs pred_ridge, sombrear zona > max(y_train_rv)
# TODO: RMSE de cada modelo restringido a los dias donde y_test_rv > max(y_train_rv)

### Preguntas de Interpretación — Parte 4

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿El Random Forest mantiene un AUC competitivo frente a Logistic Regression en la ventana walk-forward que contiene marzo 2020? ¿Qué te dice esto sobre la diferencia entre "predecir la ocurrencia de un evento de cola" (clasificación) y "predecir la magnitud de una variable en una crisis" (regresión, parte 4.2)?

> *Completa aquí*

**b)** Reporta los 4 máximos calculados en 4.2 (train, test, pred RF, pred Ridge). ¿El Ridge sobre o subestimó el pico real de volatilidad? ¿Es preferible ese comportamiento al de RF en un contexto de gestión de riesgo (piensa en el costo de subestimar el VaR)?

> *Completa aquí*

**c)** Si tuvieras que poner en producción un modelo de volatilidad que deba seguir funcionando durante la próxima crisis desconocida, ¿usarías RF, Ridge, o algún esquema híbrido (por ejemplo, RF con un tope/floor explícito, o un ensemble de ambos)? Justifica con lo observado.

> *Completa aquí*

---
## Parte 5 — Bonus: Quantile Regression Forest para Intervalos de Confianza *(15 pts)*

### Contexto

Un `RandomForestRegressor` estándar solo te da el promedio de los árboles. Pero cada árbol individual ya produce una predicción — la **distribución completa** de esas $B$ predicciones para un mismo punto $x$ contiene información sobre la incertidumbre, no solo sobre el valor esperado. Un **Quantile Regression Forest** (Meinshausen, 2006) usa esa distribución para construir intervalos de predicción: en vez de promediar las $B$ predicciones, se calculan sus percentiles empíricos.

Esto es directamente relevante para gestión de riesgo: en vez de un forecast puntual de volatilidad, puedes reportar un intervalo `[P5, P95]` — un insumo mucho más honesto para decisiones de sizing o de VaR.

### 5.1 — Implementación de Quantile Regression Forest *(10 pts)*

Implementa `qrf_predict_intervals(rf_model, X, quantiles)` que, para un `RandomForestRegressor` **ya entrenado**, recolecta la predicción de **cada árbol individual** (`rf_model.estimators_`) sobre `X`, y calcula los percentiles empíricos pedidos a través de los árboles (no a través de las observaciones).

**Requisitos:**
1. Implementar la función y aplicarla al `RandomForestRegressor` de volatilidad de la Parte 4.2 (reentrenado si quieres con más árboles, ej. `n_estimators=300`), calculando los percentiles 5, 50 y 95
2. Sobre la ventana de test (2020-01-01 a 2020-07-01), calcula la **cobertura empírica** del intervalo `[P5, P95]`: la fracción de días donde `y_test_rv` verdadero cae dentro del intervalo. Un intervalo bien calibrado en un período "normal" debería tener cobertura cercana a 90%
3. Grafica la serie real junto con la banda `[P5, P95]` durante la ventana de test, y comenta qué ocurre con la cobertura durante el shock de marzo 2020 en relación a la limitación de extrapolación vista en 4.2

In [ ]:
def qrf_predict_intervals(rf_model, X, quantiles=(0.05, 0.5, 0.95)):
    '''
    Quantile Regression Forest: usa la distribucion de predicciones de los
    arboles individuales del bosque (no el promedio) para estimar percentiles.

    Parameters
    ----------
    rf_model  : RandomForestRegressor ya entrenado
    X         : datos sobre los que predecir
    quantiles : tupla de percentiles (0-1) a calcular

    Returns
    -------
    dict {quantile: array de predicciones} de shape (n_samples,) cada una
    '''
    # TODO: recolecta las predicciones de rf_model.estimators_ sobre X (shape n_arboles x n_samples)
    # TODO: calcula np.quantile a lo largo del eje de arboles para cada quantile pedido
    pass


# === Verificacion y cobertura empirica ===
# Descomenta cuando hayas implementado la funcion

# rf_q = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
# rf_q.fit(X_train_rv, y_train_rv)
# q_preds = qrf_predict_intervals(rf_q, X_test_rv, quantiles=(0.05, 0.5, 0.95))

# coverage_90 = np.mean((y_test_rv >= q_preds[0.05]) & (y_test_rv <= q_preds[0.95]))
# print(f"Cobertura empirica del intervalo [P5, P95] durante 2020 H1: {coverage_90:.2%}")

# fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(dates_test_rv, y_test_rv, "k-", lw=1.5, label="RV realizada")
# ax.fill_between(dates_test_rv, q_preds[0.05], q_preds[0.95], alpha=0.25, color="steelblue",
#                  label="Intervalo QRF [P5, P95]")
# ax.plot(dates_test_rv, q_preds[0.5], color="steelblue", lw=1.5, ls="--", label="Mediana QRF")
# ax.set_title(f"Quantile Regression Forest — Volatilidad {TICKER}, 2020 H1")
# ax.legend(); plt.tight_layout(); plt.show()

### 5.2 — Preguntas de Interpretación *(5 pts)*

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿Qué cobertura empírica obtuviste para el intervalo `[P5, P95]` durante el shock de marzo 2020? ¿Es consistente con la limitación de extrapolación de RF vista en la Parte 4.2? Explica la conexión.

> *Completa aquí*

**b)** Si un equipo de riesgo usara este intervalo QRF como insumo directo para un límite de VaR durante una crisis desconocida, ¿qué error sistemático cometerían? Propón una mitigación concreta (por ejemplo, combinar con un modelo paramétrico, ensanchar el intervalo con un factor de seguridad, etc.)

> *Completa aquí*

---
## Tabla de Puntajes

| Parte | Descripción | Pts |
|---|---|---|
| 1.1 | `bagging_forest_fit` + `bagging_forest_predict` + verificación OOB≈1-1/e + comparación vs sklearn | 13 |
| 1.2 | `oob_score_manual` + verificación vs `oob_score_` + gráfico convergencia OOB vs test | 12 |
| 2.1 | Curvas sesgo-varianza (árbol vs RF) sobre datos reales + gap train-test | 10 |
| 2.2 | Grid search walk-forward (heatmap) + correlación OOB vs AUC walk-forward + selección final | 15 |
| 3.1 | `permutation_importance_manual` (error < 0.01 vs sklearn) + gráfico MDI vs permutation + ranking `ruido_continuo` | 13 |
| 3.2 | Importancia temporal con intervalos de confianza (multi-semilla) + heatmap + $CV_j$ | 12 |
| 4.1 | Target de eventos de cola + walk-forward RF vs Logistic (≥6 ventanas, incluye COVID) + gráfico | 13 |
| 4.2 | RF vs Ridge en extrapolación de volatilidad (máximos + RMSE en zona de extrapolación) + gráfico | 12 |
| Preguntas de interpretación (4 bloques × 3 preguntas) | Incluidas en cada parte | — |
| **Total** | | **100** |
| 5.1 (Bonus) | `qrf_predict_intervals` + cobertura empírica + gráfico banda de predicción | 10 |
| 5.2 (Bonus) | Preguntas de interpretación | 5 |
| **Total con Bonus** | | **115** |